# 🔬 第 44 课 · Attention Kernel

> naive O(N²) 与分块实现;看懂 kernel 的 q/k/v 分块与 loop;用 Triton 真写并运行一个可编译内核

**章节**: 第 7 章 · Attention Kernel 实战

---

> **📚 本笔记本属于《VLLM_learn: 图解 vLLM 推理引擎》系列**
> 风格参照《鸢尾花书》: 重图解、重直觉、循序渐进、动手实操

## 🎯 学习目标

- 理解 attention kernel 的骨架:Q 固定一行块,沿 K/V 的 N 方向循环,在线 softmax 累积输出
- 手写 naive 与分块两版实现,并数值验证两者等价
- 理解 Triton 是什么:Python DSL,编译成 CUDA 的编译管线(AST→Triton-IR→Triton-GPU→LLVM→PTX→SASS)
- 逐行读懂 Triton attention kernel:tl.program_id / tl.dot / tl.load / tl.store / tl.arange
- 在本机真实编译并运行一个 Triton attention kernel,并与 torch SDPA / naive 对比误差
- 在 GPU 上对比 naive / 分块 / Triton / SDPA 的耗时,看懂 kernel 化的收益
- 建立与 vLLM triton_attn 后端的联系

## 📑 本课目录

1. **直觉:工厂流水线与并行工人** — kernel = 许多线程并行处理 Q 的不同行块
2. **naive 实现:把打分矩阵摊开** — 三步落盘,O(N²) 显存
3. **分块实现:q 块沿 k/v 循环** — 在线 softmax + 输出累加,O(N) 显存
4. **数值验证:分块 vs naive** — 最大误差应 ~1e-6
5. **Triton:Python DSL 与编译管线** — 从源码到 SASS 的完整链路
6. **逐行讲解 + 真跑一个 Triton kernel** — program_id / tl.dot / 在线 softmax,并与 SDPA 对比
7. **真机耗时对比** — naive / 分块 / Triton / SDPA 随 N 的变化
8. **与 vLLM 的关系** — triton_attn.py 后端与 torch SDPA
9. **小结 + 练习 + 延伸阅读** — 要点、动手题、论文链接

## 🔗 参考资料

- [Triton: An Intermediate Language and Compiler for Tiled Neural Network Computations (MAPL'19)](https://arxiv.org/abs/1910.01791)
- [Triton 官方教程: Fused Attention](https://triton-lang.org/main/getting-started/tutorials/06-fused-attention.html)
- [FlashAttention (NeurIPS 2022)](https://arxiv.org/abs/2205.14135)
- [PyTorch Blog: Triton Kernel Compilation Stages](https://pytorch.org/blog/triton-kernel-compilation-stages/)

## 1. 直觉:工厂流水线与并行工人 🏭

把 attention 想象成一家工厂:每一行输出 $O_i$ 都是「第 $i$ 个查询 $q_i$ 与**所有键值**对话后」的结果。这个「对话」对每一行都是**独立**的,所以工厂可以安排**很多工人并行**——每个工人负责 $Q$ 的一小块行,沿 $K/V$ 方向从头扫到尾。

真实 GPU kernel 就是这么干的:把 $Q$ 切成一块块 **block_M**,每一块交给一组线程(program)处理;每个 program 内部再沿 $K/V$ 的 **block_N** 循环。这就是 attention kernel 的全部骨架。

> 📄 Triton 正是把「分块计算」这一层抽象出来的语言:Tillet et al.,「Triton: An Intermediate Language and Compiler for Tiled Neural Network Computations」[MAPL'19 (arXiv:1910.01791)](https://arxiv.org/abs/1910.01791)。

✅ 每课第一段代码:设置 KMP 保护、固定 seed。有 CUDA 就走真实 GPU;下面的 naive/分块/Triton 都是 GPU 上真跑、真计时。

In [ ]:
# -*- coding: utf-8 -*-
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # Windows OMP 库冲突防护
import math, time, torch, numpy as np

dev = "cuda" if torch.cuda.is_available() else "cpu"     # 有 CUDA 就跑真实 GPU（RTX 5060）
torch.manual_seed(0)
print("torch  :", torch.__version__)
print("device :", dev, "| GPU:", torch.cuda.get_device_name(0) if dev == "cuda" else "无(CPU)")
print("num_threads :", torch.get_num_threads())
print("SDPA flash 可用         :", torch.backends.cuda.flash_sdp_enabled())
print("SDPA mem_efficient 可用 :", torch.backends.cuda.mem_efficient_sdp_enabled())

## 2. naive 实现:把打分矩阵摊开 🧾

先写最直观的版本:`QK^T` 得到完整 $N \times N$ 打分矩阵 $S$,softmax 得到 $P$,再乘 $V$。三步都把**大张量写进内存**,简单但浪费(O(N²) 显存)。

**naive_attention**:把 QKᵀ / softmax / 加权和 三个大张量全部写进 HBM。

In [2]:
def naive_attention(Q, K, V):
    """朴素注意力:把 QK^T / softmax / 加权和 三个大张量全部写进 HBM。
    Q, K, V: (B, H, N, d)。返回与输入同形状的输出 O。
    """
    scale = 1.0 / math.sqrt(Q.shape[-1])          # 1/√d 缩放因子
    S = torch.einsum("bhnd,bhmd->bhnm", Q, K) * scale   # ① QK^T: (B,H,N,N)
    P = torch.softmax(S, dim=-1)                  # ② softmax: (B,H,N,N)
    O = torch.einsum("bhnm,bhmd->bhnd", P, V)     # ③ 加权和: (B,H,N,d)
    return O

## 3. 分块实现:q 块沿 k/v 循环 🔁

分块版本不再整体算出 $S$。外层循环**沿 $K/V$ 的 N 方向**一次读一块 $K_j, V_j$;对每一块,用**在线 softmax**(第 43 课)更新 running max $m$、running sum $l$,并把当前块的贡献**加权累加**进输出 $O$。这样 $S/P$ 只活在块内,$O$ 和 $m/l$ 是唯一需要保留的累加器。

**flash_attention_chunked**:外层循环沿 K/V 分块扫描,内层做 online softmax + 输出累加。这就是 kernel 的数学结构。

In [3]:
def flash_attention_chunked(Q, K, V, block_M=64):
    """分块 FlashAttention(torch 手写版):online softmax + K/V 分块扫描。
    只在寄存器/片上保存 running max(m)、running sum(l) 与累加器 O,
    大张量 S/P 从不整体落进 HBM——这就是 FlashAttention 的核心思想。
    Q, K, V: (B, H, N, d)。block_M 控制 K/V 方向上每次读入的块宽。
    """
    B, H, N, d = Q.shape
    O = torch.zeros_like(Q)                                   # 输出累加器
    m = torch.full((B, H, N, 1), float("-inf"), device=Q.device)  # 逐行 running max
    l = torch.zeros((B, H, N, 1), device=Q.device)                # 逐行 running sum
    scale = 1.0 / math.sqrt(d)
    for j in range(0, N, block_M):                            # 沿 K/V 分块扫描
        Kj = K[:, :, j:j + block_M, :]                        # 读一块 K (B,H,Bm,d)
        Vj = V[:, :, j:j + block_M, :]                        # 读一块 V (B,H,Bm,d)
        S = torch.einsum("bhnd,bhmd->bhnm", Q, Kj) * scale    # 块内打分 (B,H,N,Bm)
        m_new = torch.maximum(m, S.max(dim=-1, keepdim=True).values)   # 更新 running max
        P = torch.exp(S - m_new)                              # 以新 max 归一(防溢出)
        l_new = l * torch.exp(m - m_new) + P.sum(dim=-1, keepdim=True)  # 更新 running sum
        O = O * torch.exp(m - m_new) + torch.einsum("bhnm,bhmd->bhnd", P, Vj)  # 校正旧累加
        m = m_new                                             # 滚到下一块
        l = l_new
    return O / l                                              # 最终除以分母

## 4. 数值验证:分块 vs naive ✅

分块版的数学应等价于 naive。随机生成 Q/K/V,对比两者输出:

🎯 误差在 1e-6 量级:分块实现在数值上与标准 attention 等价。

In [ ]:
# 正确性验证:分块实现 ≡ naive 实现
B, H, N, d = 1, 4, 256, 64                       # 小规模便于快速验证
Q = torch.randn(B, H, N, d, device=dev)          # (B,H,N,d)
K = torch.randn(B, H, N, d, device=dev)
V = torch.randn(B, H, N, d, device=dev)

O_naive = naive_attention(Q, K, V)               # naive 输出 (B,H,N,d)
O_flash = flash_attention_chunked(Q, K, V, block_M=64)   # 分块输出 (B,H,N,d)
err = (O_naive - O_flash).abs().max().item()     # 最大绝对误差
print(f"分块 vs naive 最大绝对误差 = {err:.2e} @ {dev}")

## 5. Triton:Python DSL 与编译管线 🚀

工业界常用 **Triton** 写这种 kernel。Triton 不是「另一种编程语言」,而是一层 Python DSL:你写一个 `@triton.jit` 装饰的 Python 函数,描述**分块**如何计算,Triton 把它编译成 CUDA C++/PTX再交给 NVIDIA 编译器生成 SASS。

编译管线([PyTorch Blog: Triton Kernel Compilation Stages](https://pytorch.org/blog/triton-kernel-compilation-stages/)):

```
Python @triton.jit 源码
  → Triton-IR(ttir,机器无关)
  → Triton-GPU IR(ttgir,分块/布局)
  → LLVM-IR
  → PTX
  → SASS
```

关键:整个过程由 Python 的 AST 驱动,`tl.dot` 会被映射到 GPU 的 **Tensor Core(MMA)**——这是它比朴素逐元素实现快的原因之一。

## 6. 逐行讲解 + 真跑一个 Triton kernel 🤖

下面这段是**非因果**的完整可运行 Triton attention kernel(去掉了掩码,专注看分块/在线 softmax 骨架)。逐行读懂关键字:

- `@triton.jit` :把下面的 Python 函数标记为 Triton 内核(会被编译成 CUDA);
- `tl.program_id(0/1)` :当前 program 负责的 Q 行块、以及 (batch, head) 的拍平索引;
- `tl.arange(0, BM)` :生成 `[0,1,...,BM)` 的整数向量,用来算行/列偏移;
- `tl.load / tl.store` :从 HBM 读/写一个块;
- `tl.dot(q, k)` :块矩阵乘(映射到 Tensor Core);
- `m_i / l_i / acc` :在线 softmax 的 running max / sum 与输出累加器,全程驻留寄存器/SRAM;
- `tl.exp / tl.maximum / tl.sum` :块内逐元素运算;
- 收尾 `acc / l_i` 并 `tl.store` 写回输出。

**对照 torch 分块版**:外层 `for j in range(0, N, block_M)` 对应这里的 `for start_n in range(...)`;`torch.maximum/exp/sum` 对应 `tl.maximum/exp/sum`;`torch.einsum(..., P, Vj)` 对应 `tl.dot(p, v, acc)`。

我们在本机**真实编译并运行**这个内核,并与 PyTorch 内置 `scaled_dot_product_attention`(SDPA)对比。

🤖 Triton 真跑成功!fp16 精度下与 SDPA 误差 ~3e-4(符合预期);与 naive 数学等价。这就是「把分块逻辑映射到 GPU 并行线程」的完整闭环。

In [ ]:
# 真实编译并运行一个 Triton attention kernel(非因果版)
import triton
import triton.language as tl
import torch.nn.functional as F

@triton.jit
def triton_fa_fwd(Q, K, V, sm_scale, O,
                  sqh, sqm, sqk, skh, skn, skk, svh, svn, svk, soh, som, sok,
                  Z, H, N, BM: tl.constexpr, BD: tl.constexpr, BN: tl.constexpr):
    # 每个 program 处理 Q 的一行块 BM;off_hz 是 (batch,head) 的拍平索引
    start_m = tl.program_id(0); off_hz = tl.program_id(1)
    # 本块 Q 的行号、以及 K/V 方向的列号
    offs_m = start_m * BM + tl.arange(0, BM)
    offs_n = tl.arange(0, BN)
    # 用指针算术定位 Q 本块:行偏移 offs_m×sqm,列偏移 dim×sqk
    q = tl.load(Q + off_hz * sqh + offs_m[:, None] * sqm + tl.arange(0, BD)[None, :] * sqk)
    q = (q * sm_scale).to(tl.float16)                 # 预缩放,转 fp16 供 Tensor Core
    # K/V 的起始指针(后续在循环里下移)
    kp = K + off_hz * skh + tl.arange(0, BD)[:, None] * skk + offs_n[None, :] * skn
    vp = V + off_hz * svh + offs_n[:, None] * svn + tl.arange(0, BD)[None, :] * svk
    m_i = tl.full([BM], float("-inf"), dtype=tl.float32)   # running max
    l_i = tl.zeros([BM], dtype=tl.float32)                 # running sum
    acc = tl.zeros([BM, BD], dtype=tl.float32)             # 输出累加器
    for _ in range(0, tl.cdiv(N, BN) * BN, BN):            # 沿 K/V 分块扫描
        k = tl.load(kp).to(tl.float16)                     # 读一块 K (BD, BN)
        v = tl.load(vp).to(tl.float16)                     # 读一块 V (BN, BD)
        qk = tl.dot(q, k)                                  # 块内 QK^T (BM, BN)
        m_ij = tl.maximum(m_i, tl.max(qk, 1))              # 更新 running max
        p = tl.exp(qk - m_ij[:, None])                     # 以新 max 归一
        l_i = l_i * tl.exp(m_i - m_ij) + tl.sum(p, 1)      # 校正 running sum
        acc = acc * tl.exp(m_i - m_ij)[:, None]            # 校正旧输出
        acc = tl.dot(p.to(tl.float16), v, acc)             # 加权累加
        m_i = m_ij
        kp += BN * skn; vp += BN * svn                     # 指针下移一块
    acc = acc / l_i[:, None]                               # 最终除以分母
    tl.store(O + off_hz * soh + offs_m[:, None] * som + tl.arange(0, BD)[None, :] * sok,
             acc.to(O.dtype.element_ty))                   # 写回输出

def triton_flash_attention(Q, K, V, bm=64, bn=64):
    """Triton attention 的 host 包装:分配输出、转 fp16、启动 kernel。"""
    B, H, N, d = Q.shape
    O = torch.empty_like(Q)                                # 输出 (B,H,N,d)
    Qf = Q.to(torch.float16); Kf = K.to(torch.float16); Vf = V.to(torch.float16)   # Tensor Core 用 fp16
    triton_fa_fwd[(triton.cdiv(N, bm), B * H)](            # 网格:(N/bm 行块, B*H 个头)
        Qf, Kf, Vf, 1.0 / math.sqrt(d), O,
        Qf.stride(1), Qf.stride(2), Qf.stride(3),          # 各维 stride,用于指针算术
        Kf.stride(1), Kf.stride(2), Kf.stride(3),
        Vf.stride(1), Vf.stride(2), Vf.stride(3),
        O.stride(1), O.stride(2), O.stride(3),
        B * H, H, N, BM=bm, BD=d, BN=bn)                   # constexpr 块参数
    return O

torch.manual_seed(0)
q = torch.randn(1, 4, 512, 64, device=dev)         # (B,H,N,d)
k = torch.randn(1, 4, 512, 64, device=dev)
v = torch.randn(1, 4, 512, 64, device=dev)
print(f"输入: Q/K/V shape = {tuple(q.shape)} @ {dev}")
o_tri = triton_flash_attention(q, k, v)            # 编译并运行 Triton kernel
ref = F.scaled_dot_product_attention(q, k, v)      # torch 内置 SDPA 作参照
o_naive = naive_attention(q, k, v)                 # naive 作数学真值
print(f"Triton kernel 编译并真跑成功")
print(f"triton(fp16) vs torch-SDPA 最大误差 : {(o_tri - ref).abs().max().item():.2e}")
print(f"naive 与 SDPA 误差                 : {(o_naive - ref).abs().max().item():.2e}")

## 7. 真机耗时对比:naive / 分块 / Triton / SDPA ⏱️

在 GPU 上扫序列长度 $N$,用 `torch.cuda.synchronize()` 精确计时,比较四条实现:naive、torch 分块、**Triton kernel**、以及 torch 内置 SDPA。naive 随 $N$ 二次方爬升,分块/Triton/SDPA 相对平缓——这就是 kernel 化的价值。

🎯 栏目从左到右:naive → torch 分块 → Triton → SDPA。N 越大,naive 的斜率越陡,其余三条相对平缓——访存优化与 kernel 化在长序列上放大。

In [ ]:
# GPU 上扫 N,对比四种实现的真实耗时
def bench(fn, *a, iters=5):
    for _ in range(2):            # warmup
        fn(*a)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        fn(*a)
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters * 1000   # 毫秒

Ns = [256, 512, 1024, 2048]
t_naive, t_flash, t_tri, t_sdpa = [], [], [], []
print(f"{'N':>6} {'naive':>10} {'分块':>10} {'triton':>10} {'SDPA':>10}  (ms)")
for n in Ns:
    q = torch.randn(1, 4, n, 64, device=dev)     # (B,H,N,d)
    k = torch.randn(1, 4, n, 64, device=dev)
    v = torch.randn(1, 4, n, 64, device=dev)
    t_naive.append(bench(naive_attention, q, k, v))               # naive
    t_flash.append(bench(flash_attention_chunked, q, k, v, 64))   # torch 分块
    t_tri.append(bench(triton_flash_attention, q, k, v))          # Triton kernel
    t_sdpa.append(bench(lambda a, b, c: F.scaled_dot_product_attention(a, b, c), q, k, v))
    print(f"{n:>6} {t_naive[-1]:>10.3f} {t_flash[-1]:>10.3f} {t_tri[-1]:>10.3f} {t_sdpa[-1]:>10.3f}")
    torch.cuda.empty_cache()

📊 红色(naive)向上翘,蓝色/绿色/Triton 与 SDPA 相对平缓——kernel 化(含 Triton)收益随 N 放大。

In [ ]:
# 画成折线图,直观对比四条曲线的斜率
import plotly.io as pio
pio.renderers.default = "notebook"
import plotly.graph_objects as go

fig = go.Figure()
for name, arr, col in [("naive", t_naive, "#E45756"), ("torch 分块", t_flash, "#4C78A8"),
                        ("Triton", t_tri, "#54A24B"), ("SDPA(torch)", t_sdpa, "#72B7B2")]:
    fig.add_trace(go.Scatter(x=Ns, y=arr, name=name, mode="lines+markers",
                             line=dict(color=col, width=2.5)))
fig.update_layout(title="naive vs 分块 vs Triton vs SDPA 耗时(GPU, H=4, d=64)",
                  xaxis_title="序列长度 N", yaxis_title="耗时(ms)", height=440,
                  legend=dict(orientation="h", y=1.12), margin=dict(l=10, r=10, t=60, b=10))
fig

## 8. 与 vLLM 的关系 🔗

本课写的是「数学上的分块」;工程上,vLLM 把同一逻辑封装成多个**后端(backend)**(第 47 课详讲):

- `FLASH_ATTN`:调 flash-attention 库的 CUDA 内核(FA2/FA3/FA4);
- `TRITON_ATTN`:用 Triton 写的 attention 内核(工程路径 `vendor/vllm/vllm/v1/attention/backends/triton_attn.py`);
- `TORCH_SDPA`:PyTorch 内置 `scaled_dot_product_attention`,本课已用它作参照。

PyTorch 的 `F.scaled_dot_product_attention` 内部会自动调度到 FlashAttention 等内核——所以普通用户不需要手写 Triton 也能享受 kernel 化的收益。

> 📄 你在本课亲手写的 `triton_fa_fwd`,结构与 triton 官方教程的[06-fused-attention](https://triton-lang.org/main/getting-started/tutorials/06-fused-attention.html) 一致,只是去掉了因果掩码与两阶段优化。它是 vLLM triton 后端与 flash-attention 库内核的「最小教学版」。

## 9. 🖥️ Streamlit 动态演示:拖序列长度跑小实验 🎛️

运行同目录下的 `app_44_attention.py`,拖动**序列长度 / 头数**,程序会在你的 GPU 上**实时跑一小段计时实验**,对比 naive 与分块的耗时、峰值内存,并验证两者误差:

```bash
python -m streamlit run app_44_attention.py
```

浏览器打开 **http://localhost:8501**。建议把序列长度从 128 拖到 1024,看 naive 耗时如何快速爬升。完整源码如下(与同目录 app 一字不差):

📜 这就是 app_44_attention.py 的完整源码,notebook 与 app 共用同一套 naive_attention / flash_chunked 实现,保证演示与讲解一致。

In [ ]:
%%writefile app_44_attention.py
# -*- coding: utf-8 -*-
# app_44_attention.py — 从零手写 attention:naive vs 分块的时间/内存对比 🔬
import time
import numpy as np
import plotly.graph_objects as go
import streamlit as st
import torch

st.set_page_config(page_title="🔬 44 · Attention Kernel", layout="wide")
st.title("🔬 第 44 课 · Attention Kernel:naive vs 分块")

st.markdown("""
一个 attention kernel 到底在干什么?就是 **Q 的一行块,沿 K/V 的 N 方向分块扫描**,
边算 softmax 边累积输出。naive 实现把整个 N×N 打分矩阵摊在内存里;分块实现则一次只
处理一小块,内存占用小、且更接近真实 GPU kernel 的结构。下方拖动**序列长度**,实时跑一小段
实验,对比两者的耗时与峰值内存。
""")

def naive_attention(Q, K, V):
    d = Q.shape[-1]
    S = torch.einsum("bhnd,bhmd->bhnm", Q, K) / (d ** 0.5)
    P = torch.softmax(S, dim=-1)
    return torch.einsum("bhnm,bhmd->bhnd", P, V)

def flash_chunked(Q, K, V, block_M=64):
    B, H, N, d = Q.shape
    O = torch.zeros_like(Q)
    m = torch.full((B, H, N, 1), float("-inf"))
    l = torch.zeros((B, H, N, 1))
    scale = d ** -0.5
    for j in range(0, N, block_M):
        Kj = K[:, :, j:j + block_M, :]; Vj = V[:, :, j:j + block_M, :]
        S = torch.einsum("bhnd,bhmd->bhnm", Q, Kj) * scale
        m_new = torch.maximum(m, S.max(dim=-1, keepdim=True).values)
        P = torch.exp(S - m_new)
        l_new = l * torch.exp(m - m_new) + P.sum(dim=-1, keepdim=True)
        O = O * torch.exp(m - m_new) + torch.einsum("bhnm,bhmd->bhnd", P, Vj)
        m, l = m_new, l_new
    return O / l

def bench(fn, *a):
    fn(*a)
    t0 = time.perf_counter()
    for _ in range(3):
        fn(*a)
    return (time.perf_counter() - t0) / 3 * 1000

with st.sidebar:
    st.header("🎛️ 参数")
    N = st.slider("序列长度 N", 128, 1024, 256, 64)
    H = st.slider("头数 H", 1, 8, 4, 1)
    d = st.selectbox("头维度 d", [32, 64], index=0)
    st.caption("真实实验会在你的 GPU 上跑几轮计时,请耐心等待 1-2 秒。")

torch.manual_seed(0)
Q = torch.randn(1, H, N, d)
K = torch.randn(1, H, N, d)
V = torch.randn(1, H, N, d)

t_naive = bench(naive_attention, Q, K, V)
t_flash = bench(flash_chunked, Q, K, V, 64)
mem_naive = naive_attention(Q, K, V).element_size() * (H * N * N + 3 * H * N * d + H * N * d)
mem_flash = naive_attention(Q, K, V).element_size() * (3 * H * N * d + H * N * d)

c1, c2, c3, c4 = st.columns(4)
c1.metric("naive 耗时", f"{t_naive:.1f} ms")
c2.metric("分块耗时", f"{t_flash:.1f} ms")
c3.metric("耗时比(naive/分块)", f"{t_naive/max(t_flash,1e-9):.1f}×")
c4.metric("峰值内存:naive→分块", f"{mem_naive/1e6:.0f}→{mem_flash/1e6:.1f} MB")

O1 = naive_attention(Q, K, V); O2 = flash_chunked(Q, K, V, 64)
err = float((O1 - O2).abs().max())
st.metric("分块 vs naive 最大误差", f"{err:.2e}")

Ns = list(range(128, 1025, 128))
tn, tf = [], []
for n in Ns:
    q = torch.randn(1, H, n, d); k = torch.randn(1, H, n, d); v = torch.randn(1, H, n, d)
    tn.append(bench(naive_attention, q, k, v))
    tf.append(bench(flash_chunked, q, k, v, 64))

fig = go.Figure()
fig.add_trace(go.Scatter(x=Ns, y=tn, name="naive", mode="lines+markers",
                         line=dict(color="#E45756", width=2.5)))
fig.add_trace(go.Scatter(x=Ns, y=tf, name="分块 flash", mode="lines+markers",
                         line=dict(color="#4C78A8", width=2.5)))
fig.update_layout(title=f"naive vs 分块耗时随 N 变化(H={H}, d={d})",
                  xaxis_title="序列长度 N", yaxis_title="耗时(ms)", height=440,
                  legend=dict(orientation="h", y=1.12),
                  margin=dict(l=10, r=10, t=60, b=10))
st.plotly_chart(fig, use_container_width=True)

st.caption("⭐ 观察:naive 曲线随 N 二次方爬升(慢),分块相对平缓;两者结果最大误差在 1e-6 以下,"
           "说明分块实现在数学上等价于标准 attention。")

st.markdown("""
> 💡 **结论**:attention kernel 的本质是 **q 块沿 k/v 循环 + 在线 softmax 累积输出**。
> naive 把 N×N 打分矩阵整体摊开(显存 O(N²)),分块把它压回 O(N)——这正是真实 GPU kernel
> (FlashAttention / PagedAttention)的结构雏形。Triton 只是把这段逻辑映射到 GPU 并行线程而已。
""")
st.caption("《VLLM_learn: 图解 vLLM 推理引擎》第 7 章 · 第 44 课配套演示")

## ✅ 本课小结

- attention kernel 骨架 = Q 的行块 × 沿 K/V 的 N 方向循环 + 在线 softmax 累积输出
- naive 把 N×N 打分矩阵整体摊开(O(N²) 显存);分块把它压回 O(N)
- 分块实现在数值上与标准 attention 等价(误差 ~1e-6)
- Triton 是 Python DSL:program_id / tl.dot / tl.load 描述分块计算,经 Triton-IR→LLVM→PTX→SASS 编译成 CUDA
- tl.dot 映射到 Tensor Core(MMA);本机真跑 Triton kernel 与 SDPA 误差 ~3e-4(fp16)
- GPU 真测:naive 随 N 二次方爬升,Triton/SDPA 相对平缓,kernel 化收益随 N 放大
- vLLM 的 TRITON_ATTN / FLASH_ATTN / TORCH_SDPA 都是同一逻辑的工程封装

## ✍️ 动手练习

1. 把 flash_attention_chunked 的 block_M 改成 32 / 128,观察耗时与精度的变化
2. 给 flash_attention_chunked 加一个下三角因果掩码,并与 F.scaled_dot_product_attention(is_causal=True) 对比
3. 把 triton_fa_fwd 的 BM/BN 改成 32 或 128,重跑误差与耗时,观察块大小对性能的影响
4. 给 triton_fa_fwd 加因果掩码:qk = tl.where(offs_m[:,None] >= start_n+offs_n[None,:], qk, -inf)
5. 在 app_44 里加一块「误差随 block 大小变化」的折线图

## 🔗 延伸阅读

- [Triton 论文 (MAPL'19)](https://arxiv.org/abs/1910.01791)
- [Triton 官方教程: Fused Attention](https://triton-lang.org/main/getting-started/tutorials/06-fused-attention.html)
- [PyTorch Blog: Triton Kernel Compilation Stages](https://pytorch.org/blog/triton-kernel-compilation-stages/)
- [FlashAttention 论文](https://arxiv.org/abs/2205.14135)

---
> 🎉 恭喜完成本课!下一课见!